# Extracción de variables espectrales para el modelo de calidad de agua

Este notebook documenta la etapa de extracción de variables espectrales a partir de un ortomosaico multibanda y un conjunto de puntos de muestreo de campo.

El proceso consiste en:

1. cargar un raster multibanda correspondiente al ortomosaico del humedal,
2. cargar los puntos de muestreo definidos para el modelo,
3. extraer reflectancias promedio en una ventana alrededor de cada punto,
4. calcular índices espectrales derivados,
5. unir dichos resultados con los parámetros de calidad de agua medidos en laboratorio,
6. y generar un archivo espacial de salida con la información integrada.

La lógica principal de este procedimiento se encuentra implementada en el archivo `src/calidad_agua/extraccion_reflectancia.py`, mientras que este notebook actúa como interfaz documentada para su uso y trazabilidad.

## Objetivo

Construir un archivo espacial integrado que contenga, para cada punto de muestreo:

- reflectancias promedio por banda,
- índices espectrales derivados,
- y los parámetros fisicoquímicos medidos en laboratorio,

de forma que dicho producto pueda ser utilizado posteriormente en las etapas de preparación del dataset, entrenamiento, evaluación y aplicación del modelo.

In [1]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd()

while not (BASE_DIR / "src").exists() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent

sys.path.append(str(BASE_DIR))

print("BASE_DIR detectado:")
print(BASE_DIR)

BASE_DIR detectado:
C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales


## Importación de la función principal

En esta sección se importa la lógica principal del script `extraccion_reflectancia.py`, ubicado en el directorio `src/calidad_agua/`.

Este enfoque permite mantener la lógica técnica central fuera del notebook, favoreciendo la modularidad, la reutilización y la trazabilidad del flujo de trabajo.

In [2]:
from src.calidad_agua.extraccion_reflectancia import main

## Insumos requeridos

Para ejecutar correctamente este proceso, el repositorio debe contener al menos los siguientes archivos de entrada:

- un ortomosaico multibanda del humedal,
- un shapefile de puntos de muestreo,
- un archivo Excel con los resultados de laboratorio.

Para mantener la estructura del proyecto, los archivos deben ubicarse en rutas relativas como las siguientes:

- `data/raw/calidad_agua/ortomosaicos/`
- `data/raw/calidad_agua/puntos_muestreo/`
- `data/raw/calidad_agua/laboratorio/`

La salida generada por este procedimiento se almacena en:

- `data/processed/calidad_agua/`

## Parámetros configurables del script

La configuración principal del proceso se define dentro del archivo `src/calidad_agua/extraccion_reflectancia.py`.

Entre los parámetros que se pueden ajustar directamente en dicho archivo se encuentran:

- `raster_path`: ruta del ortomosaico multibanda,
- `points_path`: ruta del shapefile de puntos,
- `excel_path`: ruta del archivo Excel con resultados de laboratorio,
- `output_path`: ruta del archivo espacial de salida,
- `window_size`: tamaño de la ventana de extracción alrededor de cada punto,
- `band_names`: nombres asignados a las bandas del raster.

Para poder adaptar el trabajo con nuevos humedales, nuevas campañas o nuevos insumos sin modificar la lógica interna del procesamiento.

## Consideraciones técnicas sobre la extracción

La extracción de valores se realiza sobre una ventana cuadrada centrada en cada punto de muestreo. En lugar de tomar únicamente el valor de un píxel, se calcula el promedio de los píxeles contenidos en dicha ventana.

Este enfoque puede ayudar a reducir la sensibilidad a ruido espacial, desalineaciones geométricas pequeñas o variaciones locales del raster. No obstante, el tamaño de ventana debe ser evaluado de acuerdo con la resolución espacial del ortomosaico, el tamaño del objeto de interés y la naturaleza del fenómeno analizado.

Por esta razón, el parámetro `window_size` se deja configurable para que pueda ser ajustado según las características del caso de estudio.

## Índices espectrales calculados

Además de las reflectancias promedio por banda, el script calcula un conjunto de índices espectrales derivados a partir de las bandas del sensor multiespectral.

Estos índices permiten resumir relaciones entre bandas y pueden aportar información útil para modelar variables de calidad de agua. La selección definitiva de predictores para el modelo no se define en este notebook, sino en la etapa posterior de preparación del dataset.

## Uso del dataset integrado en el flujo multimodelo

El archivo espacial generado en esta etapa constituye la base común para el entrenamiento de los modelos de calidad de agua.

Este producto no se genera de forma independiente para cada algoritmo. Una vez extraídas las reflectancias, calculados los índices espectrales y unidos los datos de laboratorio, el mismo dataset integrado puede ser utilizado posteriormente por los modelos:

- Support Vector Regression (SVR),
- Gradient Boosting Regressor (GBR),
- Random Forest Regressor (RFR).

Por tanto, esta etapa se ejecuta una sola vez por conjunto de datos, campaña o parámetro de análisis, y sus salidas son reutilizadas en las etapas de preparación, entrenamiento, evaluación e inferencia espacial.

## Ejecución del proceso

La siguiente celda ejecuta la función principal del script de extracción. Al finalizar, se generará un archivo espacial de salida que contendrá tanto la información espectral derivada del ortomosaico como los parámetros de laboratorio asociados a cada punto.

In [4]:
main()

Cargando archivos...
Archivo Excel no encontrado. Se asumirá que los atributos requeridos ya están incluidos en el shapefile de puntos.
Raster con 6 bandas detectadas.
Calculando reflectancia promedio e índices (ventana 11x11)...
⚠️ El CRS de los puntos no coincide con el del raster.
   Se reproyectarán los puntos al CRS del raster.


Extrayendo reflectancias e índices: 100%|██████████| 78/78 [00:02<00:00, 33.42it/s]



Columnas del GeoDataFrame antes de guardar:
['Name', 'Blue', 'Green', 'Pan', 'Red', 'RedEdge', 'NIR', 'NDVI', 'NDWI', 'ID_Muestra', 'DQO', 'pH', 'Fosfatos', 'CE', 'Turbidez', 'Nitratos', 'Sulfatos', 'ficocianin', 'Chl', 'geometry']

No se realizó unión con Excel. Se conservarán los atributos originales del shapefile de puntos junto con las reflectancias e índices calculados.

✅ Archivo generado con éxito:
   C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\data\processed\calidad_agua\Puntos_Muestreo_Reflectancia_Indices_Excel_2.gpkg
   Contiene reflectancias promedio, índices espectrales y parámetros de calidad de agua.


## Producto generado

El resultado de esta etapa es un archivo espacial enriquecido que integra, por punto de muestreo:

- reflectancias promedio,
- índices espectrales,
- y variables medidas en laboratorio.

Este archivo constituye la base de trabajo para la siguiente etapa del flujo: la preparación del dataset para modelado multimodelo. A partir de este producto se construyen las matrices de entrenamiento que serán utilizadas por SVR, Gradient Boosting Regressor y Random Forest Regressor.

## Siguiente etapa del flujo

Una vez generado el archivo integrado, el siguiente paso consiste en preparar el dataset de modelado. Esto incluye:

- seleccionar la variable objetivo,
- definir las variables predictoras,
- limpiar registros incompletos,
- y, si se considera pertinente, aplicar transformaciones como el logaritmo natural sobre variables seleccionadas.

Esta fase se documenta en el notebook:

`02_preparacion_dataset_modelo.ipynb`